# QG-ORS experiments on Google Colab

Notebook này giữ cùng workflow với `RSIAT_Colab.ipynb`: chạy source trên SSD tạm của Colab, giữ CIFAR-100, log và checkpoint trên Google Drive, chạy smoke trước rồi mới bật full experiment. RSIAT baseline **không được train lại**; cell so sánh sẽ đọc kết quả RSIAT đã chạy trước đó từ Drive nếu tìm thấy.

Repo private được nạp mặc định bằng `SOURCE_MODE = 'upload_zip'`. ZIP có thể chứa source trực tiếp hoặc nằm trong một thư mục gốc như `QG-RSA-main/`; không cần đóng gói `data/datasets`, `logs` hoặc `ckpt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess, zipfile

SOURCE_MODE = 'upload_zip'  # upload_zip | github
REPO_URL = 'https://github.com/PhThuan-tech/RSIAT.git'
BRANCH = 'main'
OUTPUT_ROOT = Path('/content/drive/MyDrive/QGORS_runs')
RSIAT_OUTPUT_ROOT = Path('/content/drive/MyDrive/RSIAT_runs')
PERSISTENT_DATA_DIR = Path('/content/drive/MyDrive/RSIAT_data/datasets')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_DATA_DIR.mkdir(parents=True, exist_ok=True)

def safe_extract_zip(zip_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise ValueError(f'Unsafe ZIP entry: {member.filename}') from exc
        archive.extractall(destination)

def find_project_root(extracted_root):
    candidates = sorted({
        path.parent.resolve()
        for path in extracted_root.rglob('main.py')
        if (path.parent / 'models').is_dir()
        and (path.parent / 'exps').is_dir()
    })
    if len(candidates) != 1:
        raise RuntimeError(
            f'Expected one RSIAT project in ZIP, found {len(candidates)}: {candidates}'
        )
    return candidates[0]

if SOURCE_MODE == 'upload_zip':
    from google.colab import files
    print('Chọn đúng một file ZIP chứa repo QG-RSA...')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise ValueError(f'Expected exactly one .zip file, received: {list(uploaded)}')
    zip_name = zip_names[0]
    zip_path = Path('/content') / Path(zip_name).name
    zip_path.write_bytes(uploaded[zip_name])
    extracted_root = Path('/content/qgors_uploaded_repo')
    if extracted_root.exists():
        shutil.rmtree(extracted_root)
    extracted_root.mkdir(parents=True)
    safe_extract_zip(zip_path, extracted_root)
    PROJECT_DIR = find_project_root(extracted_root)
elif SOURCE_MODE == 'github':
    PROJECT_DIR = Path('/content/QG-RSA')
    if (PROJECT_DIR / '.git').exists():
        subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    raise ValueError(f'Unknown SOURCE_MODE: {SOURCE_MODE}')

local_data_dir = PROJECT_DIR / 'data' / 'datasets'
if local_data_dir.is_symlink():
    if local_data_dir.resolve() != PERSISTENT_DATA_DIR.resolve():
        local_data_dir.unlink()
elif local_data_dir.exists():
    if any(local_data_dir.iterdir()):
        shutil.copytree(local_data_dir, PERSISTENT_DATA_DIR, dirs_exist_ok=True)
    shutil.rmtree(local_data_dir)
local_data_dir.parent.mkdir(parents=True, exist_ok=True)
if not local_data_dir.exists():
    local_data_dir.symlink_to(PERSISTENT_DATA_DIR, target_is_directory=True)

os.chdir(PROJECT_DIR)
print('Source mode:', SOURCE_MODE)
print('Project:', Path.cwd())
print('Dataset:', PERSISTENT_DATA_DIR)
print('Outputs:', OUTPUT_ROOT)

In [ ]:
!nvidia-smi
import torch, torchvision
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('torchvision:', torchvision.__version__)

# Keep Colab's torch/torchvision pair; install only project dependencies.
%pip install -q --upgrade-strategy only-if-needed -r requirements-colab.txt
%pip install -q --upgrade-strategy only-if-needed -r requirements-qgors.txt

import timm, pennylane as qml
print('timm:', timm.__version__)
print('PennyLane:', qml.__version__)
!python -m py_compile main.py trainer.py models/base.py models/RSIAT_adapter.py models/qgors.py

test_file = Path('tests/test_qgors.py')
if test_file.exists():
    subprocess.run(['python', '-m', 'unittest', 'tests.test_qgors', '-v'], check=True)
else:
    print('Optional unit tests skipped: tests/test_qgors.py is not present in the uploaded ZIP.')

In [ ]:
# Download CIFAR-100 once to Google Drive, matching RSIAT_Colab.ipynb.
from torchvision.datasets import CIFAR100
import json

try:
    cifar_train = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=True, download=False)
    cifar_test = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=False, download=False)
    print('CIFAR-100 found in Drive; download skipped.')
except RuntimeError:
    cifar_train = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=True, download=True)
    cifar_test = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=False, download=True)

def make_colab_config(source, destination, seed=1993, resume=False):
    config = json.loads(Path(source).read_text())
    config.update({
        'seed': [seed],
        'resume': resume,
        'output_root': str(OUTPUT_ROOT),
        'device': ['0'] if torch.cuda.is_available() else [-1],
        'num_workers': 2,
        'stats_num_workers': 2,
        'pin_memory': torch.cuda.is_available(),
        'persistent_workers': True,
    })
    Path(destination).write_text(json.dumps(config, indent=2))

make_colab_config('exps/adapter_cifar224_qgors_smoke.json', '/content/qgors_smoke.json')
make_colab_config('exps/adapter_cifar224_classical_gate.json', '/content/classical_full.json', resume=True)
make_colab_config('exps/adapter_cifar224_qgors.json', '/content/qgors_full.json', resume=True)
print('CIFAR train/test:', len(cifar_train), len(cifar_test))

## QG-ORS smoke test

Smoke chạy task 0 và task 1 với 1 epoch mỗi stage. Task 1 mới kiểm tra residual projector, quantum gate, optimizer, evaluation và checkpoint. Đây chỉ là integration test, chưa phải số liệu dùng để kết luận; RSIAT baseline cũ không cần chạy lại.

In [ ]:
!python main.py --config /content/qgors_smoke.json

## Full CIFAR-100 experiments

Hai config full đã bật `resume`: nếu Drive chưa có checkpoint thì bắt đầu từ task 0; nếu đã có thì tiếp tục checkpoint task gần nhất. Chỉ bỏ comment sau khi QG-ORS smoke thành công. Classical gate là control cần thiết để tách lợi ích adaptive gating khỏi lợi ích riêng của quantum circuit.

In [ ]:
# !python main.py --config /content/classical_full.json
# !python main.py --config /content/qgors_full.json

In [ ]:
# Compare completed full runs. RSIAT is read from the output produced by
# RSIAT_Colab.ipynb; it is never retrained by this notebook.
def load_latest_result(output_root, prefix):
    run_dir = output_root / 'ckpt' / prefix / 'cifar224' / '10_10' / 'seed_1993'
    checkpoints = sorted(
        run_dir.glob('task_*.pkl'),
        key=lambda path: int(path.stem.split('_')[-1]),
    )
    if not checkpoints:
        return {'status': 'not found', 'expected_directory': str(run_dir)}
    path = checkpoints[-1]
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    curve = checkpoint.get('cnn_curve', {}).get('top1', [])
    return {
        'status': 'found',
        'top1_curve': curve,
        'average_top1': sum(curve) / len(curve) if curve else None,
        'final_top1': curve[-1] if curve else None,
        'checkpoint': str(path),
    }

results = {
    'RSIAT (previous run)': load_latest_result(RSIAT_OUTPUT_ROOT, 'all'),
    'Classical gate': load_latest_result(OUTPUT_ROOT, 'classical_gate'),
    'QG-ORS': load_latest_result(OUTPUT_ROOT, 'qgors'),
}
results

In [ ]:
print('Recent logs:')
for path in sorted((OUTPUT_ROOT / 'logs' / 'adapter').rglob('*.log'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)
print('Recent checkpoints:')
for path in sorted((OUTPUT_ROOT / 'ckpt').rglob('task_*.pkl'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)